In [0]:
%pip install databricks-vectorsearch==0.60 -q
dbutils.library.restartPython()

In [0]:
%sql
USE workspace.sephora_products_and_skincare_reviews;

ALTER TABLE bronze_pdf_chunked
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM bronze_pdf_chunked LIMIT 5;

In [0]:
import mlflow.deployments
import json

In [0]:
deploy_client = mlflow.deployments.get_deploy_client("databricks")

question = "How Sephora transforms global pricing strategy"
response = deploy_client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": [question]}
)

embeddings = [
    e["embedding"]
    for e
    in  response.data
]

In [0]:
print("Embedding shape:", len(embeddings[0]))
print("Embeddings for question:", embeddings[0])


In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

vector_search_endpoint = "sephora"
index_name = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_chunked_index"
table_name = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_chunked"

vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=table_name,
    primary_key="id",
    embedding_source_column="chunk",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)

In [0]:
index = vsc.get_index(index_name=index_name)

In [0]:
print(
    json.dumps(
        index.describe(), 
        indent=4
))

In [0]:
query_text = "What is Sephora's pricing strategy?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3
)

display(results)

In [0]:
query_text = "What is Sephora's pricing strategy?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="hybrid",
    num_results=5
)

display(results)

In [0]:
query_text = "Pricing strategy"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="FULL_TEXT",
    num_results=5
)

display(results)

In [0]:
query_text = "Pricing strategy"
results_filtered = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3,
    filters={"path LIKE": "dbfs:/Volumes/workspace/sephora_products_and_skincare_reviews/data/pdf/Sephora.pdf"}
)

display(results_filtered)

In [0]:
from databricks.vector_search.reranker import DatabricksReranker

query_text = "Pricing strategy"
results_filtered_reranked = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3,
    filters={"path LIKE": "dbfs:/Volumes/workspace/sephora_products_and_skincare_reviews/data/pdf/Sephora.pdf"},
    reranker=DatabricksReranker(columns_to_rerank=["chunk"]),
)

display(results_filtered_reranked)